# terra — Remote state, risk scoring, state diff

Demonstrates the W3 surface area:

- `terra.load.state_s3` (mocked here with `moto`)
- `terra.load.state_cached` — Parquet round-trip cache
- `terra.risk.score`, `blast_radius`, `user_data_diff`
- `terra.frame.state_diff` — compare two state files

Runs entirely on committed fixtures — no AWS credentials, no Terraform binary.

In [ ]:
from pathlib import Path
import terra

FIXTURES = Path('../tests/fixtures')

## 1. Remote state via S3 (mocked)

`terra.load.state_s3` reads `terraform.tfstate` out of an S3 backend. Below we use
`moto` to stand up an in-process S3 so the notebook is self-contained; in real use
you'd just call `terra.load.state_s3('my-bucket', 'prod/terraform.tfstate')`.

In [ ]:
import boto3
from moto import mock_aws

raw = (FIXTURES / 'state.json').read_bytes()

with mock_aws():
    s3 = boto3.client('s3', region_name='us-east-1')
    s3.create_bucket(Bucket='tf-state-demo')
    s3.put_object(Bucket='tf-state-demo', Key='prod/terraform.tfstate', Body=raw)

    state = terra.load.state_s3('tf-state-demo', 'prod/terraform.tfstate')

terra.frame.resources_df(state)

## 2. Parquet cache round-trip

Large state files are slow to JSON-parse. `state_cached` writes a Parquet copy on the
first call and skips re-parsing on subsequent calls if the cache is fresher than the
source file.

In [ ]:
import tempfile
tmp = Path(tempfile.mkdtemp())
cache = tmp / 'state.parquet'

state = terra.load.state_cached(FIXTURES / 'state.json', cache)
print('cache exists:', cache.exists(), '|', cache.stat().st_size, 'bytes')

# Second call hits the cache (no re-parse).
state2 = terra.load.state_cached(FIXTURES / 'state.json', cache)
print('round-trip equal:', state == state2)

## 3. Risk scoring

`terra.risk.score` applies a rule pack to a changes DataFrame and adds `risk` and
`risk_reasons` columns. Default rules flag stateful deletes, no-CBD replaces, IAM
widening, `user_data` mutations, and tag-only updates.

In [ ]:
plan = terra.load.plan_json(FIXTURES / 'plan.json')
changes = terra.frame.changes_df(plan)
scored = terra.risk.score(changes)
scored[['address', 'actions', 'risk', 'risk_reasons']]

In [ ]:
# High-risk only
scored.query("risk == 'high'")[['address', 'risk_reasons']]

## 4. Blast radius

Given a dependency graph and a resource address, `blast_radius` returns the set of
resources that depend on it — the things that get destroyed or replaced if it goes.

In [ ]:
g = terra.graph.from_state(state)
list(g.nodes)[:5]

In [ ]:
# Pick a node that has dependents in the fixture (subnet depends on the VPC).
terra.risk.blast_radius(g, 'module.networking.aws_vpc.main')

## 5. user_data diff

`user_data` mutations on EC2 instances force replacement. `terra.risk.user_data_diff`
decodes (base64 / gzip) before/after blobs and emits a unified diff per affected row,
so the actual cloud-init delta shows up in the notebook instead of after `apply`.

In [ ]:
diffs = terra.risk.user_data_diff(changes)
for entry in diffs:
    print(f"--- {entry['address']} ({entry['field']}) ---")
    print(entry['unified_diff'])
if not diffs:
    print('(no user_data changes in this fixture)')

## 6. State diff between two applies

`terra.frame.state_diff(before, after)` compares two State objects and returns a
DataFrame of added / removed / changed resources. Useful for post-apply review or
for diffing `terraform.tfstate.backup` against `terraform.tfstate`.

In [ ]:
# For demo purposes, diff the fixture state against itself with one resource removed.
before = terra.load.state_local(FIXTURES / 'state.json')
after = terra.load.state_local(FIXTURES / 'state.json')
after.values.root_module.resources = after.values.root_module.resources[:-1]

terra.frame.state_diff(before, after)